# Clase 3 — Datos multimodales y embeddings

## Pregunta central

> **¿Cómo puede el mismo software trabajar con tablas, texto, imágenes, audio y datos satelitales?**

## Idea principal

Cada modalidad necesita una representación numérica y un preprocesamiento coherente; un embedding resume contenido en un vector.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Reconocer la forma numérica de cinco modalidades.
- Distinguir datos crudos, features y embeddings.
- Interpretar dimensión y similitud coseno.
- Construir una búsqueda semántica pequeña.
- Comparar imágenes y representaciones de audio sin mezclar espacios.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | De datos reales a tensores |
| 2 | Features frente a embeddings |
| 3 | Similitud coseno |
| 4 | Búsqueda semántica |
| 5 | Embeddings de imagen y audio |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en el orden propuesto.
2. Antes de modificar código, observá y describí el resultado.
3. Cambiá solamente las variables marcadas con `TODO`.
4. No es necesario implementar algoritmos desde cero.
5. Si aparece un término nuevo, buscá primero su definición en el glosario de la clase.

**Conexión con el programa:** embeddings y recuperación del Track Salud, y representaciones visuales/multibanda del Track Imagen.

## Glosario mínimo

| Término | Explicación breve |
|---|---|
| Modalidad | Tipo de dato: tabla, texto, imagen, audio o ráster |
| Representación | Forma numérica entregada al software |
| Preprocesamiento | Transformación previa consistente |
| Feature | Propiedad medida o calculada |
| Embedding | Vector producido por un modelo |
| Dimensión | Cantidad de valores del vector |
| Similitud coseno | Comparación de dirección entre vectores |
| Encoder | Modelo que produce una representación |
| Retrieval | Recuperación de elementos relevantes |
| Espacio compartido | Representaciones entrenadas para ser comparables |
| Tensor | Arreglo numérico con una forma y un tipo de dato |
| Batch | Grupo de muestras procesado en una misma operación |
| Token | Fragmento en el que un tokenizador divide un texto |
| Waveform | Secuencia de amplitudes de una señal a lo largo del tiempo |
| Sample rate | Cantidad de muestras de audio por segundo |
| ASR | Reconocimiento automático de voz: transforma habla en texto |
| CRS | Sistema que da significado geográfico a unas coordenadas |
| NDVI | Índice calculado con bandas roja e infrarroja para describir contraste espectral de vegetación |
| Head | Capa final que convierte features en la salida de una tarea |
| Producto punto | Multiplicar posiciones equivalentes de dos vectores y sumar los resultados |

---
## 1. La forma de cada modalidad

Una **modalidad** es una clase de información con estructura propia.
Para una persona, una frase y una imagen son contenidos distintos.
Para una computadora, ambos deben convertirse en números antes de
llegar a un modelo.

```text
fenómeno real
    |
    v
archivo o registro
    |
    v
lectura y preprocesamiento
    |
    v
tensor numérico
    |
    v
modelo
```

La conversión debe conservar la estructura útil de cada modalidad.

| Modalidad | Representación inicial | Estructura que importa | Ejemplo de forma |
|---|---|---|---|
| Tabular | Filas y columnas | Qué significa cada columna | `(muestras, features)` |
| Texto | Secuencia de tokens | Orden y contexto | `(tokens,)` |
| Imagen RGB | Intensidades por canal | Vecindad espacial | `(alto, ancho, 3)` |
| Audio | Amplitud en el tiempo | Orden temporal y sample rate | `(muestras_audio,)` |
| Ráster multibanda | Una grilla por banda | Posición, resolución y CRS | `(bandas, alto, ancho)` |

**La forma no es un detalle:** define qué operaciones son válidas.
Una banda espectral no es una fila tabular; un token no es un píxel.

### Shape, dtype y rango

Un tensor se describe al menos con:

| Propiedad | Pregunta |
|---|---|
| Shape | ¿Cuántos ejes hay y cuánto mide cada uno? |
| Dtype | ¿Son enteros, decimales o booleanos? |
| Rango | ¿Los valores están entre 0–255, 0–1 u otra escala? |
| Semántica | ¿Qué representa cada eje y cada valor? |

Dos tensores pueden tener la misma shape y representar cosas
completamente distintas. `(4, 32, 32)` podría ser un ráster de
cuatro bandas o un batch de cuatro imágenes en escala de grises.

### Qué agrega un batch

Los modelos suelen procesar varias muestras juntas:

```text
una imagen RGB       -> (3, alto, ancho)
batch de 16 imágenes -> (16, 3, alto, ancho)
```

El primer eje pasa a indicar cuántas muestras contiene el lote.

In [ ]:
import re
import ssl

import certifi
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_sample_image

FAST_MODE = True
SEED = 42
rng = np.random.default_rng(SEED)
ssl._create_default_https_context = (
    lambda: ssl.create_default_context(cafile=certifi.where())
)

tabla = pd.DataFrame({
    "edad": [34, 58, 41],
    "consultas_previas": [1, 6, 2],
})
texto = "El lote presenta vegetación saludable."
tokens_didacticos = re.findall(
    r"\w+|[^\w\s]", texto.lower(), flags=re.UNICODE
)
imagen = load_sample_image("flower.jpg")
audio = np.sin(2 * np.pi * 220 * np.arange(16000) / 16000)
raster = rng.uniform(0, 1, size=(4, 32, 32))

resumen = pd.DataFrame([
    ["Tabular", tabla.shape, "columnas"],
    ["Texto tokenizado", (len(tokens_didacticos),), "tokens"],
    ["Imagen RGB", imagen.shape, "alto, ancho, canales"],
    ["Audio de 1 segundo", audio.shape, "muestras a 16 kHz"],
    ["Ráster 4 bandas", raster.shape, "bandas, alto, ancho"],
], columns=["modalidad", "forma", "lectura"])
resumen

## 2. Dato crudo, feature y embedding

Estas tres representaciones corresponden a momentos distintos:

```text
dato crudo -> transformación elegida -> features
                                    |
                                    v
                               modelo encoder
                                    |
                                    v
                                embedding
```

| Nivel | Quién o qué lo produce | Ejemplo |
|---|---|---|
| Dato crudo | Fuente o sensor | Píxeles, waveform, texto |
| Feature | Regla de preparación o medición | Edad, energía, NDVI |
| Embedding | Modelo encoder | Vector de 384 o 512 valores |

Una feature puede ser directamente interpretable: “edad = 42”.
Una dimensión de un embedding normalmente no tiene una explicación
aislada del tipo “el valor 17 representa turnos”. La información se
distribuye entre muchas dimensiones.

### Qué hace un encoder

Un encoder transforma una entrada en una representación:

```text
documento -> encoder de texto -> vector
imagen    -> encoder visual   -> vector
audio     -> encoder acústico -> vector
```

Contenidos parecidos para la tarea de entrenamiento suelen quedar
próximos en ese espacio. “Parecido” depende del encoder y de los
datos con los que fue entrenado.

Un embedding no es “el significado verdadero”. Depende del modelo,
sus datos de entrenamiento y la tarea para la que fue ajustado.

### Espacios compatibles e incompatibles

No se comparan directamente:

- embeddings producidos por modelos distintos;
- un embedding de 384 dimensiones con otro de 512;
- vectores de imagen y texto entrenados por separado.

Algunos modelos se entrenan para crear un **espacio compartido**.
Por ejemplo, texto e imagen pueden volverse comparables si el
entrenamiento los alineó explícitamente. La compatibilidad es una
propiedad del modelo, no del hecho de que ambos sean vectores.

## 3. Similitud coseno

Una vez que pregunta y documentos están en el mismo espacio,
necesitamos una regla para comparar vectores.

La similitud coseno compara su **dirección**, no su tamaño. Podemos
imaginar cada vector como una flecha que parte del origen:

```text
dirección parecida       dirección diferente

     b                            b
    /                            ^
   / a                          |        a ---->
```

- cercana a `1`: representaciones alineadas;
- cercana a `0`: poca relación en ese espacio;
- cercana a `-1`: direcciones opuestas.

Normalizar un vector significa ajustar su longitud a 1 sin cambiar
la dirección. Con vectores normalizados, el producto punto produce
la misma comparación que el coseno.

Para dos vectores de tres valores, el producto punto realiza:

```text
[a1, a2, a3] · [b1, b2, b3]
        =
a1*b1 + a2*b2 + a3*b3
```

### Qué no significa un score alto

Una similitud alta no verifica:

- que un documento sea verdadero;
- que contenga la respuesta completa;
- que dos casos sean equivalentes;
- que exista una relación causal;
- que el resultado sea seguro para tomar una decisión.

El score solo expresa cercanía dentro del espacio aprendido.

In [ ]:
def normalizar(vector):
    vector = np.asarray(vector, dtype=float)
    norma = np.linalg.norm(vector)
    return vector / norma if norma else vector

def similitud_coseno(a, b):
    return float(normalizar(a) @ normalizar(b))

vectores = {
    "consulta_turno": [0.9, 0.8, 0.1],
    "agenda_medica": [0.8, 0.9, 0.1],
    "imagen_satelital": [0.1, 0.0, 0.95],
}
pd.DataFrame([
    [a, b, similitud_coseno(vectores[a], vectores[b])]
    for a, b in [
        ("consulta_turno", "agenda_medica"),
        ("consulta_turno", "imagen_satelital"),
    ]
], columns=["vector_a", "vector_b", "similitud"]).round(3)

---
## 4. Experimento: búsqueda semántica de texto

Una búsqueda por palabras encuentra coincidencias literales. Una
búsqueda semántica intenta recuperar contenido relacionado aunque
la pregunta use otras palabras.

```text
pregunta: "¿puedo cambiar la fecha de mi cita?"
documento: "los turnos pueden reprogramarse..."

coincidencia literal: pocas palabras iguales
relación semántica:   intención parecida
```

El pipeline tiene dos momentos.

**Preparación de documentos:**

```text
documentos -> encoder -> embeddings -> almacenamiento
```

**Búsqueda:**

```text
pregunta -> mismo encoder -> embedding
                             |
                             v
              similitud con documentos
                             |
                             v
                      ranking top-k
```

`top-k` significa conservar los `k` resultados con mayor score.
Recuperar no es generar: el resultado de esta etapa son documentos
o fragmentos ordenados.

La práctica intenta usar un encoder preentrenado. Si no está
disponible, usa TF-IDF como fallback. TF-IDF representa términos y
frecuencia; no tiene la misma capacidad semántica, pero permite
mantener visible el contraste entre representaciones.

In [ ]:
documentos = pd.DataFrame([
    ["turnos", "Los turnos pueden reprogramarse hasta 24 horas antes."],
    ["sensores", "La calibración de sensores se realiza cada seis meses."],
    ["recetas", "Las recetas crónicas requieren validación profesional."],
    ["satelite", "Las bandas roja e infrarroja permiten calcular NDVI."],
], columns=["id", "texto"])

pregunta = "¿Con cuánta anticipación puedo cambiar un turno?"
try:
    from sentence_transformers import SentenceTransformer

    encoder_texto = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2", device="cpu"
    )
    emb_documentos = encoder_texto.encode(
        documentos["texto"].tolist(),
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    emb_pregunta = encoder_texto.encode(
        [pregunta],
        normalize_embeddings=True,
        show_progress_bar=False,
    )[0]
    modo_texto = "embeddings all-MiniLM-L6-v2"
except Exception as error:
    from sklearn.feature_extraction.text import TfidfVectorizer

    vectorizador = TfidfVectorizer(
        ngram_range=(1, 2), norm="l2"
    )
    matriz = vectorizador.fit_transform(
        documentos["texto"].tolist() + [pregunta]
    ).toarray()
    emb_documentos = matriz[:-1]
    emb_pregunta = matriz[-1]
    modo_texto = (
        "fallback TF-IDF local "
        f"({type(error).__name__})"
    )

scores = emb_documentos @ emb_pregunta
ranking = documentos.copy()
ranking["similitud"] = scores
print("Modo:", modo_texto)
ranking.sort_values("similitud", ascending=False).round(3)

### Qué observar

Leé el ranking antes de mirar solamente el score:

1. ¿El fragmento correcto aparece primero?
2. ¿Aparece dentro de los primeros `k` resultados?
3. ¿Los fragmentos recuperados contienen evidencia suficiente?
4. ¿Hay resultados cercanos pero irrelevantes?

La pregunta y el fragmento no necesitan usar exactamente las mismas
palabras. Aun así, el encoder puede acercarlos.

Retrieval puede equivocarse. Por eso se evalúa de forma separada a
cualquier LLM que use después los fragmentos. Si la evidencia
correcta no fue recuperada, el generador no la recibirá.

Métricas como `Recall@k` preguntan si el resultado relevante apareció
dentro de los primeros `k`. En esta clase hacemos la inspección de
forma manual porque el conjunto es pequeño.

---
## 5. Embeddings de imagen

Un clasificador visual puede separarse conceptualmente en dos partes:

```text
imagen
   |
   v
extractor de features -> embedding -> head de clasificación -> clases
```

ResNet18 fue entrenada para clasificación. Antes de la última capa
produce un vector de features visuales. El **head** es la capa final
que transforma ese vector en clases. Al retirar el head y
conservar el extractor, podemos reutilizar ese vector como
embedding.

La comparación visual puede responder preguntas como:

- ¿qué imagen de una galería se parece más a esta consulta?;
- ¿hay duplicados o variantes casi iguales?;
- ¿qué ejemplos debería revisar una persona?;

El concepto de parecido puede incluir color, textura, composición y
objetos. No necesariamente coincide con el criterio del negocio.

En la galería esperamos que la flor espejada y la flor con menos
color queden más cerca de la flor original que el paisaje. Si no hay
pesos preentrenados disponibles, el fallback usa histogramas RGB:
compara color, no semántica profunda.

In [ ]:
import torch
import torch.nn.functional as F
from PIL import Image, ImageEnhance, ImageOps
from torchvision.models import ResNet18_Weights, resnet18

torch.set_num_threads(min(4, torch.get_num_threads()))
flor = Image.fromarray(load_sample_image("flower.jpg")).convert("RGB")
paisaje = Image.fromarray(load_sample_image("china.jpg")).convert("RGB")
galeria = {
    "flor": flor,
    "flor_espejo": ImageOps.mirror(flor),
    "flor_sin_color": ImageEnhance.Color(flor).enhance(0.2),
    "paisaje": paisaje,
}

try:
    weights = ResNet18_Weights.DEFAULT
    modelo_imagen = resnet18(weights=weights)
    extractor = torch.nn.Sequential(
        *list(modelo_imagen.children())[:-1]
    )
    extractor.eval()
    preprocesar = weights.transforms()
    modo_imagen = "features ResNet18 preentrenada"
except Exception as error:
    extractor = None
    preprocesar = None
    modo_imagen = (
        "fallback histograma RGB local "
        f"({type(error).__name__})"
    )

def embedding_imagen(img):
    if extractor is not None:
        lote = preprocesar(img).unsqueeze(0)
        with torch.no_grad():
            vector = extractor(lote).flatten(1)
        return F.normalize(vector, dim=1)[0]

    array = np.asarray(img.resize((64, 64)), dtype=np.float32) / 255
    histograma = np.concatenate([
        np.histogram(
            array[..., canal],
            bins=16,
            range=(0, 1),
            density=True,
        )[0]
        for canal in range(3)
    ])
    vector = torch.tensor(histograma, dtype=torch.float32)
    return F.normalize(vector, dim=0)

emb_imagenes = {k: embedding_imagen(v) for k, v in galeria.items()}
fig, axes = plt.subplots(1, len(galeria), figsize=(14, 3))
for ax, (nombre, imagen_galeria) in zip(axes, galeria.items()):
    ax.imshow(imagen_galeria)
    ax.set_title(nombre)
    ax.axis("off")
plt.suptitle(f"Galería comparada — {modo_imagen}")
plt.tight_layout()
plt.show()

consulta = "flor"
similitudes = pd.DataFrame([
    [nombre, float(emb_imagenes[consulta] @ vector)]
    for nombre, vector in emb_imagenes.items()
    if nombre != consulta
], columns=["imagen", "similitud"])
similitudes.sort_values("similitud", ascending=False).round(3)

## 6. Una representación preparada de audio

Un encoder de audio puede resumir una señal completa o un fragmento:

```text
waveform -> preprocesamiento acústico -> encoder -> embedding
```

Según su entrenamiento, el embedding puede representar:

- contenido hablado;
- identidad o características de voz;
- tipo de sonido;
- ambiente acústico;
- emoción aparente.

Es importante conocer el objetivo del modelo antes de interpretar
cercanía.

Para no descargar otro encoder en esta clase usamos vectores
didácticos preparados. El “original” y su versión con ruido se
construyen deliberadamente próximos; “alarma” se genera por separado.
Estos vectores **no fueron extraídos de un audio real** y no permiten
evaluar un modelo acústico.

En la clase 6 veremos waveform, espectrograma y Whisper sobre un WAV.

In [ ]:
base_audio = normalizar(rng.normal(size=12))
embeddings_audio = {
    "consulta_original": base_audio,
    "consulta_con_ruido": normalizar(
        base_audio + rng.normal(0, 0.05, size=12)
    ),
    "alarma": normalizar(rng.normal(size=12)),
}
pd.DataFrame([
    [nombre, similitud_coseno(
        embeddings_audio["consulta_original"], vector
    )]
    for nombre, vector in embeddings_audio.items()
    if nombre != "consulta_original"
], columns=["audio", "similitud"]).round(3)

## Actividad — elegir representación y comparación

Para cada caso separá cuatro decisiones:

| Decisión | Pregunta |
|---|---|
| Modalidad | ¿Qué tipo de dato llega al sistema? |
| Representación | ¿Qué estructura debe conservarse? |
| Modelo | ¿Qué encoder o algoritmo la procesará? |
| Comparación/salida | ¿Qué necesita consumir la aplicación? |

Modificá una fila o agregá un caso. Evitá comparar embeddings de
espacios incompatibles. Si proponés un espacio compartido, indicá
qué modelo fue entrenado para alinear las modalidades.

In [ ]:
decisiones = pd.DataFrame([
    ["Buscar una política similar a una pregunta", "embedding de texto"],
    ["Encontrar imágenes visualmente cercanas", "embedding de imagen"],
    ["Medir cobertura en cada píxel", "ráster/máscara"],
    ["Transcribir una llamada", "waveform + modelo ASR"],
    ["Predecir demanda desde columnas", "features tabulares"],
], columns=["problema", "representacion"])

# TODO: agregá un caso de tu industria y justificá la representación.
decisiones["por_que"] = [
    "Pregunta y documentos deben compartir un espacio semántico.",
    "Se comparan representaciones del mismo encoder visual.",
    "La salida conserva una posición geográfica por celda.",
    "ASR recibe amplitud muestreada en el tiempo.",
    "Cada fila contiene las variables de un caso.",
]
decisiones

---

## Síntesis de la clase

- Cada modalidad tiene una forma y un preprocesamiento propios.
- Features y embeddings no son sinónimos.
- La similitud coseno compara vectores dentro de un mismo espacio.
- Embeddings permiten recuperación y reutilización de representaciones.
- Una similitud alta no garantiza una respuesta correcta.

## Comprobación conceptual

Antes de continuar, intentá responder sin mirar el notebook:

1. ¿Cuál era el problema central de la clase?
2. ¿Qué entrada recibió el sistema y qué salida produjo?
3. ¿Qué decisión humana siguió siendo necesaria?
4. ¿Qué limitación observaste en el experimento?

Si podés explicarlo con tus propias palabras y justificarlo con un resultado visible, alcanzaste el objetivo introductorio.

## Puente con la próxima clase

La clase 4 muestra cómo las redes aprenden esas representaciones y compara PyTorch con Keras.

## Conexión con los tracks

Salud usará embeddings para NLP, audio y RAG; Imagen los reutilizará para clasificación, detección y análisis multibanda.

La implementación profunda, el trabajo con datasets reales y las decisiones de producción se desarrollarán en los módulos especializados.